# 03, Modelagem de Dados (Camada Gold)

## Objetivo
Modelar os dados limpos da camada Silver em um esquema estrela, com
tabelas fato e dimensões, prontas para responder às perguntas de
negócio do MVP.

## Modelagem adotada
Esquema Estrela, com uma tabela fato central (fato_internacoes)
cercada por quatro dimensões: tempo, diagnóstico, município e hospital.

## Tabelas criadas
- dim_tempo: calendário de 2024, uma linha por dia
- dim_diagnostico: códigos CID-10 do diagnóstico principal
- dim_municipio: municípios de residência dos pacientes (código IBGE)
- dim_hospital: estabelecimentos de saúde (CNPJ)
- fato_internacoes: uma linha por AIH, com medidas e chaves das dimensões

In [0]:
# Configuração da camada Gold.
# Leitura da Silver e preparação da pasta de destino.
from pyspark.sql.functions import (col, when, datediff, year, month,
                                   dayofmonth, quarter, date_format,
                                   dayofweek, substring, floor, lit)

caminho_silver = "/Volumes/workspace/default/dados_mvp/silver/sih_rd_mg_2024"
caminho_gold = "/Volumes/workspace/default/dados_mvp/gold"

# Remoção da pasta Gold antes da reconstrução, garantindo idempotência:
# cada execução recria a camada do zero, sem acumular versões antigas
# nem gerar incompatibilidade de schema entre execuções.
dbutils.fs.rm(caminho_gold, recurse=True)

df_silver = spark.read.format("delta").load(caminho_silver)
print("Linhas na Silver:", df_silver.count())

In [0]:
# Dimensão tempo: calendário de 2024, uma linha por dia.
# A chave (sk_tempo) é a própria data. Permite análises por mês,
# trimestre e dia da semana, respondendo à pergunta de sazonalidade.
# Os nomes de mês e dia da semana são mapeados para português para
# não depender do idioma configurado no cluster.
df_tempo = (spark.sql("""
    SELECT explode(sequence(to_date('2024-01-01'), to_date('2024-12-31'), interval 1 day)) AS dt
""")
    .withColumn("sk_tempo", col("dt"))
    .withColumn("ano", year(col("dt")))
    .withColumn("mes", month(col("dt")))
    .withColumn("dia", dayofmonth(col("dt")))
    .withColumn("trimestre", quarter(col("dt")))
    .withColumn("nome_mes",
        when(col("mes") == 1, "Janeiro")
        .when(col("mes") == 2, "Fevereiro")
        .when(col("mes") == 3, "Março")
        .when(col("mes") == 4, "Abril")
        .when(col("mes") == 5, "Maio")
        .when(col("mes") == 6, "Junho")
        .when(col("mes") == 7, "Julho")
        .when(col("mes") == 8, "Agosto")
        .when(col("mes") == 9, "Setembro")
        .when(col("mes") == 10, "Outubro")
        .when(col("mes") == 11, "Novembro")
        .otherwise("Dezembro"))
    .withColumn("dia_semana",
        when(dayofweek(col("dt")) == 1, "Domingo")
        .when(dayofweek(col("dt")) == 2, "Segunda-feira")
        .when(dayofweek(col("dt")) == 3, "Terça-feira")
        .when(dayofweek(col("dt")) == 4, "Quarta-feira")
        .when(dayofweek(col("dt")) == 5, "Quinta-feira")
        .when(dayofweek(col("dt")) == 6, "Sexta-feira")
        .otherwise("Sábado"))
    .select("sk_tempo", "ano", "mes", "dia", "trimestre", "nome_mes", "dia_semana"))

df_tempo.write.mode("overwrite").format("delta").save(f"{caminho_gold}/dim_tempo")
print("dim_tempo:", df_tempo.count(), "linhas")

In [0]:
# Dimensão diagnóstico: um registro por código CID-10 do diagnóstico
# principal. Conforme a definição do projeto, apenas o código é
# mantido, com o capítulo CID derivado da primeira letra do código.
df_diag = (df_silver.select("DIAG_PRINC").distinct()
    .withColumnRenamed("DIAG_PRINC", "codigo_cid10")
    .withColumn("capitulo_cid", substring(col("codigo_cid10"), 1, 1))
    .withColumn("sk_diagnostico", col("codigo_cid10"))
    .select("sk_diagnostico", "codigo_cid10", "capitulo_cid"))

df_diag.write.mode("overwrite").format("delta").save(f"{caminho_gold}/dim_diagnostico")
print("dim_diagnostico:", df_diag.count(), "linhas")

In [0]:
# Dimensão município: município de residência do paciente, com o
# código IBGE de 7 dígitos. A UF é derivada dos dois primeiros
# dígitos do código IBGE.
df_mun = (df_silver.select("MUNIC_RES").distinct()
    .withColumnRenamed("MUNIC_RES", "codigo_ibge")
    .withColumn("uf", substring(col("codigo_ibge"), 1, 2))
    .withColumn("sk_municipio", col("codigo_ibge"))
    .select("sk_municipio", "codigo_ibge", "uf"))

df_mun.write.mode("overwrite").format("delta").save(f"{caminho_gold}/dim_municipio")
print("dim_municipio:", df_mun.count(), "linhas")

In [0]:
# Dimensão hospital: estabelecimento de saúde onde ocorreu a
# internação, identificado pelo CNPJ (campo CGC_HOSP do arquivo RD).
df_hosp = (df_silver.select("CGC_HOSP").distinct()
    .withColumnRenamed("CGC_HOSP", "cnpj_hospital")
    .withColumn("sk_hospital", col("cnpj_hospital"))
    .select("sk_hospital", "cnpj_hospital"))

df_hosp.write.mode("overwrite").format("delta").save(f"{caminho_gold}/dim_hospital")
print("dim_hospital:", df_hosp.count(), "linhas")

In [0]:
from pyspark.sql.functions import col, when, lit, datediff

# Tabela fato: uma linha por AIH (internação). Referencia as dimensões
# pelas chaves (sk_*) e carrega as medidas: diárias, valores,
# permanência, óbito e uso de UTI.
#
# Idade: a coluna IDADE já chega convertida pelo PySUS em anos (número
# inteiro), portanto não há decodificação do formato codificado do SIH
# (unidade + valor de dois dígitos). O cast para inteiro garante o tipo
# numérico e mantém eventuais não informados como NULL, tratados como
# "Ignorado" na faixa etária.
#
# Sexo: no leiaute RD do SIH/SUS o campo SEXO usa 1 = Masculino e
# 3 = Feminino (diferente do SINASC/SIM, que usam 1 e 2). Qualquer
# outro valor cai em "Ignorado".
#
# UTI: o leiaute RD de MG/2024 não contém o campo QT_DIAS_UTI (dias
# em UTI), apenas MARCA_UTI, que indica o uso de UTI e o tipo
# (0 = não utilizou; 1 a 5 = tipos de UTI). Por isso a fato usa
# MARCA_UTI e deriva um indicador binário de uso de UTI.
df_fato = (df_silver
    .withColumn("idade_anos", col("IDADE").cast("int"))
    .withColumn("faixa_etaria",
        when(col("idade_anos").isNull(), "Ignorado")
        .when(col("idade_anos") < 5, "0-4")
        .when(col("idade_anos") < 10, "5-9")
        .when(col("idade_anos") < 20, "10-19")
        .when(col("idade_anos") < 30, "20-29")
        .when(col("idade_anos") < 40, "30-39")
        .when(col("idade_anos") < 50, "40-49")
        .when(col("idade_anos") < 60, "50-59")
        .when(col("idade_anos") < 70, "60-69")
        .when(col("idade_anos") < 80, "70-79")
        .otherwise("80+"))
    .withColumn("sexo_desc",
        when(col("SEXO") == 1, "Masculino")
        .when(col("SEXO") == 3, "Feminino")
        .otherwise("Ignorado"))
    .withColumn("permanencia_dias", datediff(col("DT_SAIDA"), col("DT_INTER")))
    .withColumn("indicador_obito", when(col("MORTE") == 1, 1).otherwise(0))
    .withColumn("indicador_uti", when(col("MARCA_UTI").cast("int") > 0, 1).otherwise(0))
    .withColumn("sk_tempo", col("DT_INTER"))
    .withColumn("sk_diagnostico", col("DIAG_PRINC"))
    .withColumn("sk_municipio", col("MUNIC_RES"))
    .withColumn("sk_hospital", col("CGC_HOSP"))
    .select("sk_tempo", "sk_diagnostico", "sk_municipio", "sk_hospital",
            "N_AIH", "sexo_desc", "idade_anos", "faixa_etaria",
            "QT_DIARIAS", "MARCA_UTI", "indicador_uti",
            "VAL_SH", "VAL_SP", "VAL_SADT", "VAL_TOT",
            "permanencia_dias", "indicador_obito"))

df_fato.write.mode("overwrite").format("delta").save(f"{caminho_gold}/fato_internacoes")
print("fato_internacoes:", df_fato.count(), "linhas")

# Validação pós-gravação: conferir que idade e sexo foram decodificados
print("idade_ignorada:", df_fato.filter(col("faixa_etaria") == "Ignorado").count())
print("sexo_ignorado:", df_fato.filter(col("sexo_desc") == "Ignorado").count())

In [0]:
# Célula G8: Registro das tabelas Gold no Unity Catalog como tabelas gerenciadas
# O catálogo e o schema reais são detectados do ambiente, evitando nomes fixos.
# Tabelas gerenciadas copiam os dados para o storage do catálogo e aparecem
# no Catalog Explorer.
# Observação: CREATE TABLE com LOCATION dentro de /Volumes não é permitido
# no Unity Catalog, por isso o registro é feito via saveAsTable.

catalogo = spark.catalog.currentCatalog()
schema = spark.catalog.currentDatabase()
print("Catálogo detectado:", catalogo)
print("Schema detectado:", schema)

tabelas = ["dim_tempo", "dim_diagnostico", "dim_municipio", "dim_hospital", "fato_internacoes"]

# Bloco 1: remove as tabelas gerenciadas antigas do catálogo.
# Motivo: a fato_internacoes foi registrada em execução anterior com o schema
# antigo (idade_anos como decimal). Depois da correção da G7, a pasta nova tem
# idade_anos como inteiro e o overwrite não consegue mesclar os dois esquemas,
# abortando com DELTA_FAILED_TO_MERGE_FIELDS. O DROP recria a tabela do zero,
# sem conflito de schema. As dimensões também são removidas para manter a
# reexecução limpa e idempotente.
for tabela in tabelas:
    spark.sql(f"DROP TABLE IF EXISTS {catalogo}.{schema}.{tabela}")
    print("Tabela removida:", f"{catalogo}.{schema}.{tabela}")

# Bloco 2: registra as 5 tabelas com o schema atual das pastas Delta da Gold.
for tabela in tabelas:
    df_tabela = spark.read.format("delta").load(f"{caminho_gold}/{tabela}")
    df_tabela.write.mode("overwrite").saveAsTable(f"{catalogo}.{schema}.{tabela}")
    print("Tabela registrada:", f"{catalogo}.{schema}.{tabela}", "-", df_tabela.count(), "linhas")

# Bloco 3: reaplica as descrições das tabelas e colunas (comentários) no
# Unity Catalog, porque o DROP apaga toda a documentação existente.

spark.sql(f"COMMENT ON TABLE {catalogo}.{schema}.fato_internacoes IS 'Tabela fato das internacoes hospitalares do SUS em Minas Gerais, 2024, uma linha por AIH.'")
spark.sql(f"COMMENT ON TABLE {catalogo}.{schema}.dim_tempo IS 'Calendario de 2024, uma linha por dia.'")
spark.sql(f"COMMENT ON TABLE {catalogo}.{schema}.dim_diagnostico IS 'Dimensao de diagnosticos, um registro por codigo CID-10.'")
spark.sql(f"COMMENT ON TABLE {catalogo}.{schema}.dim_municipio IS 'Dimensao de municipios de residencia dos pacientes.'")
spark.sql(f"COMMENT ON TABLE {catalogo}.{schema}.dim_hospital IS 'Dimensao de estabelecimentos de saude (hospitais).'")

# Comentários das colunas da fato_internacoes
for coluna, descricao in {
    "sk_tempo": "Chave da dimensao tempo (data de internacao)",
    "sk_diagnostico": "Chave da dimensao diagnostico (codigo CID-10)",
    "sk_municipio": "Chave da dimensao municipio (codigo IBGE de residencia)",
    "sk_hospital": "Chave da dimensao hospital (CNPJ)",
    "N_AIH": "Numero da AIH, identificador da internacao",
    "sexo_desc": "Sexo do paciente: Masculino, Feminino ou Ignorado",
    "idade_anos": "Idade em anos",
    "faixa_etaria": "Faixa etaria: 0-4, 5-9, 10-19, 20-29, 30-39, 40-49, 50-59, 60-69, 70-79, 80+ ou Ignorado",
    "QT_DIARIAS": "Quantidade de diarias da internacao",
    "MARCA_UTI": "Uso de UTI e tipo: 0 nao utilizou, 1 a 5 tipos de UTI",
    "indicador_uti": "Indicador binario de uso de UTI: 1 se houve uso, 0 caso contrario",
    "VAL_SH": "Valor dos servicos hospitalares da AIH",
    "VAL_SP": "Valor dos servicos profissionais da AIH",
    "VAL_SADT": "Valor dos servicos auxiliares de diagnostico e terapia",
    "VAL_TOT": "Valor total da AIH",
    "permanencia_dias": "Permanencia em dias, DT_SAIDA menos DT_INTER",
    "indicador_obito": "Indicador de obito: 1 se houve obito, 0 caso contrario",
}.items():
    spark.sql(f"COMMENT ON COLUMN {catalogo}.{schema}.fato_internacoes.{coluna} IS '{descricao}'")

# Comentários das colunas da dim_tempo
for coluna, descricao in {
    "sk_tempo": "Chave da dimensao, igual a data",
    "ano": "Ano da data (2024)",
    "mes": "Mes, de 1 a 12",
    "dia": "Dia do mes, de 1 a 31",
    "trimestre": "Trimestre, de 1 a 4",
    "nome_mes": "Nome do mes em portugues",
    "dia_semana": "Nome do dia da semana em portugues",
}.items():
    spark.sql(f"COMMENT ON COLUMN {catalogo}.{schema}.dim_tempo.{coluna} IS '{descricao}'")

# Comentários das colunas da dim_diagnostico
for coluna, descricao in {
    "sk_diagnostico": "Chave da dimensao, igual ao codigo CID-10",
    "codigo_cid10": "Codigo CID-10 do diagnostico principal",
    "capitulo_cid": "Capitulo CID derivado da primeira letra do codigo",
}.items():
    spark.sql(f"COMMENT ON COLUMN {catalogo}.{schema}.dim_diagnostico.{coluna} IS '{descricao}'")

# Comentários das colunas da dim_municipio
for coluna, descricao in {
    "sk_municipio": "Chave da dimensao, igual ao codigo IBGE de 7 digitos",
    "codigo_ibge": "Codigo IBGE do municipio de residencia",
    "uf": "UF derivada dos dois primeiros digitos do codigo IBGE",
}.items():
    spark.sql(f"COMMENT ON COLUMN {catalogo}.{schema}.dim_municipio.{coluna} IS '{descricao}'")

# Comentários das colunas da dim_hospital
for coluna, descricao in {
    "sk_hospital": "Chave da dimensao, igual ao CNPJ do estabelecimento",
    "cnpj_hospital": "CNPJ do estabelecimento (campo CGC_HOSP)",
}.items():
    spark.sql(f"COMMENT ON COLUMN {catalogo}.{schema}.dim_hospital.{coluna} IS '{descricao}'")

print("Registro e documentacao do catalogo concluidos.")

In [0]:
# Verificação do esquema estrela: contagem de cada tabela e um JOIN
# de exemplo entre fato e dimensões, provando que o modelo está
# conectado e consultável. A verificação usa a API de DataFrame,
# independente do registro no catálogo.
fato = spark.read.format("delta").load(f"{caminho_gold}/fato_internacoes")
dim_tempo = spark.read.format("delta").load(f"{caminho_gold}/dim_tempo")
dim_diag = spark.read.format("delta").load(f"{caminho_gold}/dim_diagnostico")
dim_mun = spark.read.format("delta").load(f"{caminho_gold}/dim_municipio")
dim_hosp = spark.read.format("delta").load(f"{caminho_gold}/dim_hospital")

print("dim_tempo:", dim_tempo.count(), "linhas")
print("dim_diagnostico:", dim_diag.count(), "linhas")
print("dim_municipio:", dim_mun.count(), "linhas")
print("dim_hospital:", dim_hosp.count(), "linhas")
print("fato_internacoes:", fato.count(), "linhas")

# JOIN de exemplo: 10 internações com diagnóstico, município e mês
resultado = (fato
    .join(dim_diag, fato.sk_diagnostico == dim_diag.sk_diagnostico, "left")
    .join(dim_mun, fato.sk_municipio == dim_mun.sk_municipio, "left")
    .join(dim_tempo, fato.sk_tempo == dim_tempo.sk_tempo, "left")
    .select(fato.N_AIH, dim_diag.codigo_cid10, dim_mun.codigo_ibge,
            dim_tempo.ano, dim_tempo.mes, fato.VAL_TOT, fato.permanencia_dias)
    .limit(10))
resultado.show()

# Checagem de integridade: todas as linhas da fato devem casar com a
# dimensão diagnóstico (sem órfãos)
total = fato.count()
com_chave = fato.join(dim_diag, fato.sk_diagnostico == dim_diag.sk_diagnostico, "inner").count()
print("Linhas da fato conectadas ao diagnóstico:", com_chave, "de", total)

In [0]:
# Documentação do Catálogo de Dados no Unity Catalog.
# Cada tabela e cada campo recebe uma descrição com contexto, tipo e
# domínio de valores, atendendo ao requisito de catálogo de dados do
# trabalho. As descrições ficam visíveis no Catalog Explorer e podem
# ser transcritas para o README.

# dim_tempo
spark.sql("COMMENT ON TABLE workspace.default.dim_tempo IS 'Dimensão tempo: calendário de 2024, uma linha por dia, usada para análises de sazonalidade das internações.'")
spark.sql("COMMENT ON COLUMN workspace.default.dim_tempo.sk_tempo IS 'Chave da dimensão tempo, igual à data (tipo date).'")
spark.sql("COMMENT ON COLUMN workspace.default.dim_tempo.ano IS 'Ano da data (2024).'")
spark.sql("COMMENT ON COLUMN workspace.default.dim_tempo.mes IS 'Mês da data, de 1 a 12.'")
spark.sql("COMMENT ON COLUMN workspace.default.dim_tempo.dia IS 'Dia do mês, de 1 a 31.'")
spark.sql("COMMENT ON COLUMN workspace.default.dim_tempo.trimestre IS 'Trimestre do ano, de 1 a 4.'")
spark.sql("COMMENT ON COLUMN workspace.default.dim_tempo.nome_mes IS 'Nome do mês em português (janeiro a dezembro).'")
spark.sql("COMMENT ON COLUMN workspace.default.dim_tempo.dia_semana IS 'Nome do dia da semana em português (domingo a sábado).'")

# dim_diagnostico
spark.sql("COMMENT ON TABLE workspace.default.dim_diagnostico IS 'Dimensão diagnóstico: códigos CID-10 do diagnóstico principal das internações.'")
spark.sql("COMMENT ON COLUMN workspace.default.dim_diagnostico.sk_diagnostico IS 'Chave da dimensão diagnóstico, igual ao código CID-10.'")
spark.sql("COMMENT ON COLUMN workspace.default.dim_diagnostico.codigo_cid10 IS 'Código CID-10 do diagnóstico principal (ex: I50, J18).'")
spark.sql("COMMENT ON COLUMN workspace.default.dim_diagnostico.capitulo_cid IS 'Capítulo CID derivado da primeira letra do código (A a Z).'")

# dim_municipio
spark.sql("COMMENT ON TABLE workspace.default.dim_municipio IS 'Dimensão município: municípios de residência dos pacientes, com código IBGE.'")
spark.sql("COMMENT ON COLUMN workspace.default.dim_municipio.sk_municipio IS 'Chave da dimensão município, igual ao código IBGE de 7 dígitos.'")
spark.sql("COMMENT ON COLUMN workspace.default.dim_municipio.codigo_ibge IS 'Código IBGE do município de residência (7 dígitos).'")
spark.sql("COMMENT ON COLUMN workspace.default.dim_municipio.uf IS 'UF derivada dos dois primeiros dígitos do código IBGE.'")

# dim_hospital
spark.sql("COMMENT ON TABLE workspace.default.dim_hospital IS 'Dimensão hospital: estabelecimentos de saúde onde ocorreram as internações.'")
spark.sql("COMMENT ON COLUMN workspace.default.dim_hospital.sk_hospital IS 'Chave da dimensão hospital, igual ao CNPJ do estabelecimento.'")
spark.sql("COMMENT ON COLUMN workspace.default.dim_hospital.cnpj_hospital IS 'CNPJ do estabelecimento de saúde (campo CGC_HOSP do arquivo RD).'")

# fato_internacoes
spark.sql("COMMENT ON TABLE workspace.default.fato_internacoes IS 'Fato internações: uma linha por AIH (internação SUS em MG em 2024), com medidas e chaves das dimensões.'")
spark.sql("COMMENT ON COLUMN workspace.default.fato_internacoes.sk_tempo IS 'Chave da dimensão tempo (data de internação).'")
spark.sql("COMMENT ON COLUMN workspace.default.fato_internacoes.sk_diagnostico IS 'Chave da dimensão diagnóstico (código CID-10).'")
spark.sql("COMMENT ON COLUMN workspace.default.fato_internacoes.sk_municipio IS 'Chave da dimensão município (código IBGE de residência).'")
spark.sql("COMMENT ON COLUMN workspace.default.fato_internacoes.sk_hospital IS 'Chave da dimensão hospital (CNPJ).'")
spark.sql("COMMENT ON COLUMN workspace.default.fato_internacoes.N_AIH IS 'Número da AIH, identificador da internação.'")
spark.sql("COMMENT ON COLUMN workspace.default.fato_internacoes.sexo_desc IS 'Sexo do paciente: Masculino, Feminino ou Ignorado.'")
spark.sql("COMMENT ON COLUMN workspace.default.fato_internacoes.idade_anos IS 'Idade em anos, decodificada do formato SIH.'")
spark.sql("COMMENT ON COLUMN workspace.default.fato_internacoes.faixa_etaria IS 'Faixa etária: 0-4, 5-9, 10-19, 20-29, 30-39, 40-49, 50-59, 60-69, 70-79, 80+ ou Ignorado.'")
spark.sql("COMMENT ON COLUMN workspace.default.fato_internacoes.QT_DIARIAS IS 'Quantidade de diárias da internação.'")
spark.sql("COMMENT ON COLUMN workspace.default.fato_internacoes.MARCA_UTI IS 'Marca de uso de UTI do leiaute RD: 0 = não utilizou, 1 a 5 = tipos de UTI.'")
spark.sql("COMMENT ON COLUMN workspace.default.fato_internacoes.indicador_uti IS 'Indicador binário de uso de UTI: 1 se houve uso, 0 caso contrário.'")
spark.sql("COMMENT ON COLUMN workspace.default.fato_internacoes.VAL_SH IS 'Valor dos serviços hospitalares da AIH.'")
spark.sql("COMMENT ON COLUMN workspace.default.fato_internacoes.VAL_SP IS 'Valor dos serviços profissionais da AIH.'")
spark.sql("COMMENT ON COLUMN workspace.default.fato_internacoes.VAL_SADT IS 'Valor dos serviços auxiliares de diagnóstico e terapia da AIH.'")
spark.sql("COMMENT ON COLUMN workspace.default.fato_internacoes.VAL_TOT IS 'Valor total da AIH.'")
spark.sql("COMMENT ON COLUMN workspace.default.fato_internacoes.permanencia_dias IS 'Permanência em dias, calculada como DT_SAIDA menos DT_INTER.'")
spark.sql("COMMENT ON COLUMN workspace.default.fato_internacoes.indicador_obito IS 'Indicador de óbito: 1 se houve óbito, 0 caso contrário.'")

In [0]:
# Verificação das descrições aplicadas no catálogo (evidência)
spark.sql("DESCRIBE EXTENDED workspace.default.fato_internacoes").show(100, truncate=False)
spark.sql("DESCRIBE EXTENDED workspace.default.dim_diagnostico").show(100, truncate=False)

In [0]:

catalogo = spark.catalog.currentCatalog()
schema = spark.catalog.currentDatabase()
spark.sql(f"SHOW TABLES IN {catalogo}.{schema}").show(truncate=False)

In [0]:
spark.sql(f"DESCRIBE TABLE EXTENDED {catalogo}.{schema}.dim_hospital").show(truncate=False)


In [0]:
spark.sql(f"DESCRIBE TABLE EXTENDED {catalogo}.{schema}.fato_internacoes").show(truncate=False)

In [0]:
display(dbutils.fs.ls("/Volumes/workspace/default/dados_mvp/bronze/sih_rd_mg_2024"))

In [0]:
display(dbutils.fs.ls("/Volumes/workspace/default/dados_mvp/silver/sih_rd_mg_2024"))